### Notebook collecting commands/scripts which are run after fits/scans complete

In [10]:
# imports
# autoreload magic:
%load_ext autoreload
%autoreload 2
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from scipy.stats import norm
%matplotlib inline
import sys

from NNMFit.utilities import load_pickle
from NNMFit.utilities import ScanHandler

unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
sys.path.append(unblind_script_path)
from freefit_parameter_config import FreefitParamConfig

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


run scan analysis: calculate MC Histogram, Data Histogram, and Saturated LLH of the best fit

In [14]:
# new fits with updated data, muon template, bdts
fit_path_first = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/"    
fit_path_bdt_models = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2/"    
fit_path_reproduce = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce/"    

saturated_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/saturated_final"
analysis_config_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/input_pars_only"
os.system(f"mkdir -p {saturated_path}")
os.system(f"mkdir -p {analysis_config_path}")

0

In [21]:
scans = {
    # first iteration, buggy gradient, mc, wrong muon template
    "first_iteration": 
        f"{fit_path_first}/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2",

    # problems should be fixed, stepwise
    "start": # reproduce first iteration
        f"{fit_path_reproduce}/start",
    "update_grad_mc": # update gradient and mc datasets
        f"{fit_path_reproduce}/update_grad_mc",
    "update_grad_mc_asr": # active string requirement, small change livetime
        f"{fit_path_reproduce}/update_grad_mc_asr",
    "update_grad_mc_asr_muon": # new muon template
        f"{fit_path_reproduce}/update_grad_mc_asr_muon",

    # differrent bdt models
    "FinalTopology": 
        f"{fit_path_bdt_models}/FinalTopology_double_energy_length_binning/nbestfit20_scan_astro_gamma_1D_10steps",
    "11features": 
        f"{fit_path_bdt_models}/mcd-simpletopology_flux-hese_feat-11features/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps",
    "11features_plus_rloglmilli": 
        f"{fit_path_bdt_models}/mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps",
    "11features_plus_evtgen": 
        f"{fit_path_bdt_models}/mcd-simpletopology_flux-hese_feat-11features_plus_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps",
    "11features_plus_econf": 
        f"{fit_path_bdt_models}/mcd-simpletopology_flux-hese_feat-11features_plus_econf/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps",
    "11features_plus_rloglmilli_econf_evtgen": 
        f"{fit_path_bdt_models}/mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps",

}

In [22]:
for name, scan_dir in scans.items():
    # run do_scan_analysis.py with scan_dir as arg:
    print(f"Processing scan directory: {scan_dir}")
    os.system(f"python {unblind_script_path}/do_scan_analysis.py {scan_dir} --sat_llh_save_dir {saturated_path}/{name}")

Processing scan directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
Auto-Detecting the best fit from among the freefits in the scan path.
Initializing on server cobalt-14.icecube.wisc.edu. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only
found the minimum llh value:  212.8606063267837
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
This is the index of the best fit: 17, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Freefit_17.pickle
found 

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


Data Histogram already exists: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Data_Histogram.pickle
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Data_Histogram.pickle
LLH Map already exists: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/LLH_Maps_Freefit_17.pickle
Saturated LLH: 90.96186371833632
Min LLH: 212.8606063267837, sat. LLH: 90.96186371833632
LLH info saved to: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/saturated_final/first_iteration/SatLLH.txt
Processing scan directory: /data/user/tvaneede/Glob

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


found the minimum llh value:  212.85726985147545
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//start
This is the index of the best fit: 12, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//start/Freefit_12.pickle
found the minimum llh value:  212.85726985147545
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//start
This is the index of the best fit: 12, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//start/Freefit_12.pickle
Generating Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_he

...

found the minimum llh value:  190.2588272757899
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps
This is the index of the best fit: 17, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps/Freefit_17.pickle
found the minimum llh value:  190.2588272757899
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_ga

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


found the minimum llh value:  192.11982375793787
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_econf/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps
This is the index of the best fit: 10, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_econf/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps/Freefit_10.pickle
found the minimum llh value:  192.11982375793787
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_econf/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gam

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


found the minimum llh value:  193.4985828453482
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps
This is the index of the best fit: 6, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps/Freefit_06.pickle
found the minimum llh value:  193.4985828453482
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.33333

Collect the bestfit parameter values in a config file (for later pseudoexperiments etc.)

In [23]:
# find the best-fit for each scan and write a configuration file using the
# bestfit params as input params:
freefit_config_names = {}
min_llh_dict = {}
scan_llh = {}
for name, scan_path in scans.items():
    print(scan_path)
    freefit_config_hdl = FreefitParamConfig(scan_path=scan_path)
    print(freefit_config_hdl.get_name())
    freefit_config_hdl.write(custom_name=f"{analysis_config_path}/{name}.yaml")
    llh = freefit_config_hdl.get_min_llh_freefit(freefit_config_hdl.scan_hdl)
    min_llh_dict[freefit_config_hdl.get_name()] = llh
    scan_llh[name] = llh
    freefit_config_names[os.path.basename(os.path.dirname(scan_path))
                        ] = freefit_config_hdl.get_name()


/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
Initializing on server cobalt-14.icecube.wisc.edu. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only
found the minimum llh value:  212.8606063267837
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
This is the index of the best fit: 17, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits//force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Freefit_17.pickle
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pa

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")
/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
     

found the minimum llh value:  197.27268322784008
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//update_grad_mc
This is the index of the best fit: 4, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//update_grad_mc/Freefit_04.pickle
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only/Input_params_data_fits_debug_bdt_reproduce_update_grad_mc_Freefit_04.yaml
Writing input params to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/input_pars_only/update_grad_mc.yaml
found the minimum llh value:  197.27268322784008
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_reproduce//update_g

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")
/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
     

found the minimum llh value:  190.96611970245408
Found 20 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps
This is the index of the best fit: 4, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits_debug_bdt_2//mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli/bdt1_0.333333_bdt2_0.366667_length_10/nbestfit20_scan_astro_gamma_1D_10steps/Freefit_04.pickle
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only/Input_params_bdt1_0.333333_bdt2_0.366667_length_10_nbestfit20_scan_astro_gamma_1D_10steps_Freefit_04.yaml
Writing input params to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfi

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")
/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
     

In [30]:
# For each scan, create a subfolder with full data configs (separate + combined detectors)
# and component-only variants (Astro / Conv / Muon).
import yaml
import copy

base_config_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting"
pse_config_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/pse/hese/clean"
os.system(f"mkdir -p {base_config_dir}")
os.system(f"mkdir -p {pse_config_dir}")

DETECTOR_CONFIGS_SEPARATE = [
    "IC86_pass2_SnowStorm_FTP_HESE_Cascades",
    "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades",
    "IC86_pass2_SnowStorm_FTP_HESE_Tracks",
]
DETECTOR_CONFIGS_COMBINED = ["IC86_pass2_SnowStorm_FTP_HESE_Combined"]

COMPONENT_OVERRIDES = {
    "Astro": {"conv_norm": 0, "muongun_norm": 0},
    "Conv":  {"astro_norm": 0, "muongun_norm": 0},
    "Muon":  {"astro_norm": 0, "conv_norm": 0},
}


def write_data_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: data\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")


def write_pse_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: pseudoexp\n")
        f.write("pseudoexp_type: poisson\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")

for scan_name,scan_path in scans.items():
    input_yaml_path = f"{analysis_config_path}/{scan_name}.yaml"
    with open(input_yaml_path) as f:
        base_data = yaml.safe_load(f)
    params = base_data["input_params"]

    # pse configs
    write_pse_config(params, DETECTOR_CONFIGS_SEPARATE, f"{pse_config_dir}/{scan_name}.yaml")
    print(f"Written: {pse_config_dir}/{scan_name}.yaml")

    # data configs for plotting
    scan_dir = f"{base_config_dir}/{scan_name}"
    os.makedirs(scan_dir, exist_ok=True)

    variants = [
        (scan_name,                    DETECTOR_CONFIGS_SEPARATE),
        (f"{scan_name}_combined",      DETECTOR_CONFIGS_COMBINED),
    ]

    for stem, det_configs in variants:
        write_data_config(params, det_configs, f"{scan_dir}/{stem}.yaml")
        print(f"Written: {scan_dir}/{stem}.yaml")

        for component, overrides in COMPONENT_OVERRIDES.items():
            write_data_config(params, det_configs, f"{scan_dir}/{stem}_{component}.yaml", overrides=overrides)
            print(f"Written: {scan_dir}/{stem}_{component}.yaml")

Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/pse/hese/clean/first_iteration.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/first_iteration/first_iteration.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/first_iteration/first_iteration_Astro.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/first_iteration/first_iteration_Conv.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/first_iteration/first_iteration_Muon.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/plotting/first_iteration/first_iteration_combined.yaml
Written: /data/user/tvaneede/Global